# M18 · Différentiation automatique et calcul matriciel — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m18_retropropagation/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m18_retropropagation/03_notebook.ipynb`, créée par `python tools/start_chapter.py M18`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M18.1 à M18.10, M18.15, M18.16 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M18.20 | Les nombres duaux : la classe `Dual`, et la dérivée d'une fonction d'une variable · `Dual, dual_derivative` | 🔨 | ★★ | 45 |
|  | M18.21 | Le gradient en mode direct : n passages avec des nombres duaux · `forward_gradient` | 🔨 | ★ | 20 |
|  | M18.22 | Lire le gradient dans la différentielle : G[i, j] = df(E_ij), de la forme de la variable · `gradient_from_differential` | 🔨 | ★ | 15 |
| B | M18.23 | SymPy : dériver une fonction de matrice symbolique, et comparer aux identités | 📦 | ★ | 25 |
|  | M18.24 | Prédire le nombre d'opérations de chaque mode, puis le compter | 🔮 | ★ | 10 |
|  | M18.25 | Différences finies contre nombres duaux : l'erreur selon le pas h | 🔬 | ★★ | 20 |
| C | M18.26 | Dessiner un graphe de calcul, avec les dérivées locales sur les arêtes et les adjoints sur les nœuds | 🎨 | ★ | 30 |
|  | M18.27 | Chasse à l'erreur : un gradient transposé, un adjoint compté sur un seul chemin, un pas h trop petit | 🐛 | ★ | 15 |
|  | M18.28 | Tester une classe par ses propriétés : distributivité et dérivées connues des nombres duaux | 🛠️ | ★ | 15 |
|  | M18.29 | Défi : dériver une boucle, la méthode de Newton pour √a, en mode direct | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M18")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : un graphe de calcul et ses chemins, le mode direct par les nombres duaux, le mode inverse par les adjoints, le coût des deux modes, et les gradients de fonctions d'une matrice (formes bilinéaires, traces, $\ln\det$, une moyenne de distances, une différentielle).
- Écrire les quatre fonctions de `mymath/autodiff.py` (la classe `Dual`, `dual_derivative`, `forward_gradient`, `gradient_from_differential`) et les tester.
- Comparer SymPy aux identités du calcul matriciel, prédire puis compter le coût des deux modes, mesurer l'erreur des différences centrées contre la dérivée duale, dessiner un graphe de calcul, corriger trois erreurs classiques, tester une classe par ses propriétés, et relever un défi : dériver la méthode de Newton.

**Rappel express.** Un calcul est un graphe sans cycle ; une **dérivée locale** $\frac{\partial w}{\partial u}$ sur chaque arête ; $\frac{\partial y}{\partial x}$ est la somme, sur les chemins de $x$ à $y$, des produits des dérivées locales. **Mode direct** : $\dot{w} = \sum_{u \to w}\frac{\partial w}{\partial u}\,\dot{u}$ depuis une graine ; nombres duaux, $\varepsilon^2 = 0$, $f(a + \varepsilon) = f(a) + f'(a)\,\varepsilon$. **Mode inverse** : $\bar{y} = 1$, puis, dans l'ordre inverse, $\bar{u} = \sum_{u \to w}\bar{w}\,\frac{\partial w}{\partial u}$ (on **accumule**). Pour $\mathbb{R}^n \to \mathbb{R}^m$ : $n$ passes directes, $m$ passes arrière. Gradient par rapport à une matrice : de la forme de $\mathbf{X}$ ; $\mathrm{d}f = \operatorname{tr}(\mathbf{G}^\top\mathrm{d}\mathbf{X})$, $G_{ij} = \mathrm{d}f(\mathbf{E}_{ij})$ ; $\nabla\,\mathbf{a}^\top\mathbf{X}\mathbf{b} = \mathbf{a}\mathbf{b}^\top$, $\nabla\operatorname{tr}(\mathbf{A}\mathbf{X}) = \mathbf{A}^\top$, $\nabla\operatorname{tr}(\mathbf{X}^\top\mathbf{A}\mathbf{X}) = (\mathbf{A} + \mathbf{A}^\top)\mathbf{X}$, $\nabla\ln\det\mathbf{X} = \mathbf{X}^{-\top}$. Vérifier un gradient : $\frac{f(\mathbf{X} + h\mathbf{H}) - f(\mathbf{X} - h\mathbf{H})}{2h} \approx \sum_{ij}G_{ij}H_{ij}$, $\mathbf{H}$ au hasard et pas symétrique, $h \approx 10^{-5}$.

## Partie 0 · Vérifier tes exercices papier (✏️ M18.1 à M18.10, 🔗 M18.15 et M18.16)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question (une valeur exacte se tape aussi comme un calcul, `1/8`) ;
- une liste `[0.2, -1]` ; un nombre dual $a + b\varepsilon$ s'écrit comme la liste `[a, b]` ;
- une matrice comme la liste de ses **lignes** : `[[1, 0], [-2, 3]]` ; un nombre décimal s'y écrit avec un point, `[[0.5, 1], [0, 2]]` ; une matrice d'une seule ligne garde ses deux paires de crochets ;
- une fraction irréductible entre guillemets, `"3/7"`, quand la question le dit.

Les réponses pas encore remplies affichent ⏳.

**Ex M18.1 — Le graphe de calcul d'une formule de physique, et l'ordre de ses calculs**

In [ ]:
# Ex M18.1: the values you found on paper
answer_M18_1a = ...  # a) a list of 10 numbers: [u1, ..., u10]
answer_M18_1b = ...  # b) a number
answer_M18_1c = ...  # c) an integer
answer_M18_1d = ...  # d) an integer
answer_M18_1e = ...  # e) an integer

for letter, answer in zip("abcde", [answer_M18_1a, answer_M18_1b, answer_M18_1c, answer_M18_1d, answer_M18_1e]):
    wbm.check(f"M18.1{letter}", answer)

**Ex M18.2 — Dériver par la somme sur les chemins : une variable qui sert deux fois**

In [ ]:
# Ex M18.2: the values you found on paper
answer_M18_2a = ...  # a) a number
answer_M18_2b = ...  # b) a number
answer_M18_2c = ...  # c) an irreducible fraction, as text
answer_M18_2d = ...  # d) an irreducible fraction, as text
answer_M18_2e = ...  # e) an irreducible fraction, as text
answer_M18_2f = ...  # f) an irreducible fraction, as text
answer_M18_2g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M18_2a, answer_M18_2b, answer_M18_2c, answer_M18_2d, answer_M18_2e, answer_M18_2f, answer_M18_2g]):
    wbm.check(f"M18.2{letter}", answer)

**Ex M18.3 — Le mode direct à la main : des nombres duaux sur une fonction de deux variables**

In [ ]:
# Ex M18.3: the values you found on paper
answer_M18_3a = ...  # a) a list of 2 numbers (a dual number a + b ε as [a, b])
answer_M18_3b = ...  # b) a list of 2 numbers (a dual number a + b ε as [a, b])
answer_M18_3c = ...  # c) a list of 2 numbers (a dual number a + b ε as [a, b])
answer_M18_3d = ...  # d) a list of 2 numbers (a dual number a + b ε as [a, b])
answer_M18_3e = ...  # e) a list of 2 numbers (a dual number a + b ε as [a, b])
answer_M18_3f = ...  # f) a list of 2 numbers (a dual number a + b ε as [a, b])
answer_M18_3g = ...  # g) a list of 2 numbers (a dual number a + b ε as [a, b])

for letter, answer in zip("abcdefg", [answer_M18_3a, answer_M18_3b, answer_M18_3c, answer_M18_3d, answer_M18_3e, answer_M18_3f, answer_M18_3g]):
    wbm.check(f"M18.3{letter}", answer)

**Ex M18.4 — Le mode inverse à la main : les adjoints d'un graphe de six nœuds**

In [ ]:
# Ex M18.4: the values you found on paper
answer_M18_4a = ...  # a) a list of 4 numbers: [u1, u2, u3, f]
answer_M18_4b = ...  # b) a list of 2 numbers: [ū3, ū2]
answer_M18_4c = ...  # c) a number
answer_M18_4d = ...  # d) a list of 2 numbers: [p̄, q̄]
answer_M18_4e = ...  # e) a number
answer_M18_4f = ...  # f) a list of 2 numbers (a dual number a + b ε as [a, b])

for letter, answer in zip("abcdef", [answer_M18_4a, answer_M18_4b, answer_M18_4c, answer_M18_4d, answer_M18_4e, answer_M18_4f]):
    wbm.check(f"M18.4{letter}", answer)

**Ex M18.5 — Le même gradient par les deux modes, et le compte des opérations**

In [ ]:
# Ex M18.5: the values you found on paper
answer_M18_5a = ...  # a) a number
answer_M18_5b = ...  # b) a list of 3 numbers
answer_M18_5c = ...  # c) a list of 2 numbers: [ū4, ū3]
answer_M18_5d = ...  # d) a list of 2 numbers: [ū1, ū2]
answer_M18_5e = ...  # e) a list of 3 numbers
answer_M18_5f = ...  # f) a list of 2 integers
answer_M18_5g = ...  # g) a list of 2 integers

for letter, answer in zip("abcdefg", [answer_M18_5a, answer_M18_5b, answer_M18_5c, answer_M18_5d, answer_M18_5e, answer_M18_5f, answer_M18_5g]):
    wbm.check(f"M18.5{letter}", answer)

**Ex M18.6 — Le gradient de aᵀXb, de la forme de X**

In [ ]:
# Ex M18.6: the values you found on paper
answer_M18_6a = ...  # a) a number
answer_M18_6b = ...  # b) a number
answer_M18_6c = ...  # c) a matrix (list of rows)
answer_M18_6d = ...  # d) a list of 2 integers
answer_M18_6e = ...  # e) a number
answer_M18_6f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_M18_6a, answer_M18_6b, answer_M18_6c, answer_M18_6d, answer_M18_6e, answer_M18_6f]):
    wbm.check(f"M18.6{letter}", answer)

**Ex M18.7 — Les gradients de tr(AX) et de tr(XᵀAX), coordonnée par coordonnée**

In [ ]:
# Ex M18.7: the values you found on paper
answer_M18_7a = ...  # a) a number
answer_M18_7b = ...  # b) a matrix (list of rows)
answer_M18_7c = ...  # c) a number
answer_M18_7d = ...  # d) a number
answer_M18_7e = ...  # e) a matrix (list of rows)
answer_M18_7f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_M18_7a, answer_M18_7b, answer_M18_7c, answer_M18_7d, answer_M18_7e, answer_M18_7f]):
    wbm.check(f"M18.7{letter}", answer)

**Ex M18.8 — Le gradient de log det X sur une matrice 2 × 2, à la main puis par la formule**

In [ ]:
# Ex M18.8: the values you found on paper
answer_M18_8a = ...  # a) a number
answer_M18_8b = ...  # b) a matrix (list of rows)
answer_M18_8c = ...  # c) a matrix (list of rows)
answer_M18_8d = ...  # d) a matrix (list of rows)
answer_M18_8e = ...  # e) a number
answer_M18_8f = ...  # f) a number (5 decimals)

for letter, answer in zip("abcdef", [answer_M18_8a, answer_M18_8b, answer_M18_8c, answer_M18_8d, answer_M18_8e, answer_M18_8f]):
    wbm.check(f"M18.8{letter}", answer)

**Ex M18.9 — Le gradient d'une moyenne de distances : le point qui minimise la somme des distances à des villes**

In [ ]:
# Ex M18.9: the values you found on paper
answer_M18_9a = ...  # a) a number (3 decimals)
answer_M18_9b = ...  # b) a list of 2 numbers (4 decimals)
answer_M18_9c = ...  # c) a list of 2 numbers (3 decimals)
answer_M18_9d = ...  # d) a list of 2 numbers
answer_M18_9e = ...  # e) a number (4 decimals)
answer_M18_9f = ...  # f) a number (3 decimals)

for letter, answer in zip("abcdef", [answer_M18_9a, answer_M18_9b, answer_M18_9c, answer_M18_9d, answer_M18_9e, answer_M18_9f]):
    wbm.check(f"M18.9{letter}", answer)

**Ex M18.10 — La différentielle d'une fonction de matrice, df = tr(Gᵀ dX), et le gradient G qu'on y lit**

In [ ]:
# Ex M18.10: the values you found on paper
answer_M18_10a = ...  # a) a number
answer_M18_10b = ...  # b) a matrix (list of rows)
answer_M18_10c = ...  # c) a number
answer_M18_10d = ...  # d) a number
answer_M18_10e = ...  # e) a matrix (list of rows)
answer_M18_10f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_M18_10a, answer_M18_10b, answer_M18_10c, answer_M18_10d, answer_M18_10e, answer_M18_10f]):
    wbm.check(f"M18.10{letter}", answer)

**Ex M18.15 — Pont : pour une simulation de ℝ²⁰⁰⁰ dans ℝ³, combien de passages dans chaque mode ?**

In [ ]:
# Ex M18.15: the values you found on paper
answer_M18_15a = ...  # a) a list of 2 integers
answer_M18_15b = ...  # b) a list of 2 integers
answer_M18_15c = ...  # c) a list of 2 integers
answer_M18_15d = ...  # d) an integer
answer_M18_15e = ...  # e) an integer
answer_M18_15f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_M18_15a, answer_M18_15b, answer_M18_15c, answer_M18_15d, answer_M18_15e, answer_M18_15f]):
    wbm.check(f"M18.15{letter}", answer)

**Ex M18.16 — Pont : de la convention de MML (gradient en ligne) à celle de NumPy**

In [ ]:
# Ex M18.16: the values you found on paper
answer_M18_16a = ...  # a) a matrix of one row: [[a, b]]
answer_M18_16b = ...  # b) a list of 1 integer
answer_M18_16c = ...  # c) a list of 2 lists of 2 integers
answer_M18_16d = ...  # d) a list of 2 numbers
answer_M18_16e = ...  # e) a list of 3 integers

for letter, answer in zip("abcde", [answer_M18_16a, answer_M18_16b, answer_M18_16c, answer_M18_16d, answer_M18_16e]):
    wbm.check(f"M18.16{letter}", answer)

## Partie A · Ta librairie : les nombres duaux, le gradient en mode direct, et le gradient lu dans la différentielle

Fiche §3, §5, §6 et §7. Les quatre fonctions de `mymath/autodiff.py`, leurs tests, et un usage de chacune.

In [ ]:
# Tools of the notebook (parts A to C)
import ast
import inspect
import math
import re
import textwrap
import warnings

import matplotlib.pyplot as plt
import numpy as np
import sympy
from matplotlib.text import Annotation

from wbm.nbtools import StopCheck, checked, drawn, filled, run_pytest_group, same, verdict

TEST_M18 = "tests/test_m18_autodiff.py"          # the tests of mymath/autodiff.py


class Counted:
    """A function that counts its calls: Counted(g)(p) is g(p), and .calls is the number of calls made so far."""

    def __init__(self, g):
        self.g, self.calls = g, 0

    def __call__(self, *args):
        self.calls += 1
        return self.g(*args)

### Ex M18.20 — Les nombres duaux : la classe `Dual`, et la dérivée d'une fonction d'une variable · `Dual, dual_derivative` 🔨 ★★ ⏱️ 45 min
**Objectif :** programmer l'arithmétique des nombres duaux $a + b\varepsilon$, $\varepsilon^2 = 0$, et en tirer la dérivée exacte d'une fonction d'une variable, sans pas $h$.  
**Prérequis :** fiche §3 ; workbookIA 0A (surcharge d'opérateurs) · **mymath :** `autodiff.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/autodiff.py`, lis la docstring du module puis celle de la classe ou de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. ⚠️ Le squelette n'importe rien : ajoute `import math` et `import numpy as np` en tête du fichier (`math` pour `exp`, `log`, `sin` et `cos` d'un nombre ; NumPy pour les tableaux de `forward_gradient` et de `gradient_from_differential`). SymPy est interdit : les tests comparent tes résultats aux siens, et un test vérifie que ton module ne l'importe pas. Les nombres se rendent en `float` de Python, les gradients en tableaux NumPy de flottants, de la forme de la variable.

Écris la classe `Dual` et la fonction `dual_derivative` (fiche §3), en suivant leurs docstrings :
- `Dual(value, derivative)` est le nombre dual $a + b\varepsilon$, avec $\varepsilon^2 = 0$ ; le constructeur et `__repr__` sont fournis. Écris `+`, `-`, `*` et `/` **des deux côtés** (`__add__` et `__radd__`, et ainsi de suite), l'opposé `-u`, la puissance `u ** p` par une constante, et les méthodes `exp`, `log`, `sin` et `cos`. Chaque opération rend un **nouveau** `Dual`, sans modifier ses opérandes ; un nombre ordinaire $c$ compte comme `Dual(c, 0.0)`, une constante.
- Les règles : $(a + b\varepsilon) \pm (c + d\varepsilon) = (a \pm c) + (b \pm d)\,\varepsilon$ ; $(a + b\varepsilon)(c + d\varepsilon) = ac + (ad + bc)\,\varepsilon$ ; $\frac{a + b\varepsilon}{c + d\varepsilon} = \frac{a}{c} + \frac{bc - ad}{c^2}\,\varepsilon$ (`ZeroDivisionError` si $c = 0$) ; $(a + b\varepsilon)^p = a^p + p\,a^{p-1}b\,\varepsilon$ ; $g(a + b\varepsilon) = g(a) + g'(a)\,b\,\varepsilon$ pour $g = \exp$, $\ln$, $\sin$, $\cos$ (`ValueError` pour `log` si $a \le 0$).
- La puissance : `TypeError` si l'exposant est un `Dual` ; `ValueError` si $a < 0$ et que $p$ n'est pas entier, ou si $a = 0$ et $p < 1$ avec $p \ne 0$ ; `u ** 0` vaut `Dual(1.0, 0.0)`.
- `dual_derivative(f, x)` rend le couple de flottants $(f(x), f'(x))$ en évaluant **une fois** `f(Dual(x, 1.0))` : la graine 1 dit que la dérivée de $x$ par rapport à lui-même vaut 1. Si `f` rend un nombre (une fonction constante), sa dérivée est `0.0` ; `TypeError` si elle rend autre chose.

**L'usage.** Une fonction écrite une seule fois avec les opérations de `Dual` se dérive par `dual_derivative`. Avec tes fonctions, calcule :
- a) la dérivée de $t \mapsto \frac{t^2 + 1}{t - 3}$ en 1 (4 décimales) ;
- b) la dérivée de $t \mapsto t^t$ en 2 (4 décimales) : l'exposant varie, `**` ne convient pas ; écris $t^t = e^{t\ln t}$, soit `(t * t.log()).exp()` ;
- c) la valeur et la dérivée de $t \mapsto \cosh^2 t - \sinh^2 t$ en $1{,}3$, avec $\cosh t = \frac{e^t + e^{-t}}{2}$ et $\sinh t = \frac{e^t - e^{-t}}{2}$ écrits avec `exp` (une liste de 2 nombres, 4 décimales).

Dans ta copie : retrouve a) par la règle du quotient, et b) en dérivant $e^{t\ln t}$ à la main. Pourquoi c) donne-t-elle ces deux nombres-là, en tout point ? Combien d'évaluations de $f$ `dual_derivative` fait-elle, et pourquoi la dérivée n'a-t-elle besoin d'aucun pas $h$ ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M18")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M18, "dual or module_uses")

**L'usage** : trois dérivées calculées par tes nombres duaux.

In [ ]:
answer_M18_20a = answer_M18_20b = answer_M18_20c = ...
with wbm.attempt("M18.20"):          # if you call dual_derivative here: ⏳ while it is not written
    answer_M18_20a = ...  # a) a number (4 decimals): the derivative of (t² + 1) / (t − 3) at 1
    answer_M18_20b = ...  # b) a number (4 decimals): the derivative of t^t = exp(t ln t) at 2
    answer_M18_20c = ...  # c) a list of 2 numbers (4 decimals): [value, derivative] of cosh²t − sinh²t at 1.3

In [ ]:
for letter, answer in zip("abc", [answer_M18_20a, answer_M18_20b, answer_M18_20c]):
    wbm.check(f"M18.20{letter}", answer, computed=True)

### Ex M18.21 — Le gradient en mode direct : n passages avec des nombres duaux · `forward_gradient` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer le gradient d'une fonction de $n$ variables par $n$ passes directes, une graine par passe, et en compter le coût.  
**Prérequis :** fiche §3 et §5 ; Ex M18.20 · **mymath :** `autodiff.py` · **Parcours :** C

Écris `forward_gradient(f, x)` (fiche §3 et §5), en suivant sa docstring : le gradient de $f : \mathbb{R}^n \to \mathbb{R}$ en $\mathbf{x}$ par $n$ **passes directes**. La passe $i$ évalue `f` sur la liste des duaux `Dual(x_j, 1.0 if j == i else 0.0)` : seule $x_i$ bouge (la graine $\mathbf{e}_i$), et la partie en $\varepsilon$ du résultat est $\frac{\partial f}{\partial x_i}$. `ValueError` si `x` n'est pas un tableau 1-D non vide ; une fonction qui rend un nombre (une constante) a une dérivée nulle à chaque passe ; le gradient se rend en tableau NumPy de flottants, et `x` n'est pas modifié.

**L'usage.** La cellule fournie donne la hauteur du projectile de la fiche (§1), $y(v, \theta, t) = v\,t\sin\theta - 5t^2$ (`height_21`), et sa **portée**, la distance à laquelle une balle lancée du sol retombe, $R(v, \theta) = \frac{v^2\sin(2\theta)}{10}$ (`range_21`), toutes deux écrites avec les opérations de `Dual` (`theta.sin()`). Avec ta fonction, et `Counted` (cellule des outils : `Counted(g)` se comporte comme `g` et compte ses appels dans `calls`), calcule :
- a) le gradient de la hauteur en $(v, \theta, t) = (20 ;\ \frac{\pi}{6} ;\ 0{,}5)$ (`POINT_HEIGHT_21` ; une liste de 3 nombres, 4 décimales) ;
- b) le gradient de la portée en $(v, \theta) = (20 ;\ \frac{\pi}{8})$ (`POINT_RANGE_21` ; une liste de 2 nombres, 4 décimales) ;
- c) le nombre d'appels de la fonction par `forward_gradient`, pour a) puis pour b) (une liste de 2 entiers).

Dans ta copie : compare a) aux dérivées partielles de la fiche (§2 à §4). Écris $\frac{\partial R}{\partial v}$ et $\frac{\partial R}{\partial\theta}$ à la main, et retrouve b). Pour quel angle $\frac{\partial R}{\partial\theta}$ s'annule-t-il, et que dit-il de la portée ? Combien de passes faudrait-il pour le gradient d'une fonction de 1 000 variables, en mode direct, puis en mode inverse (fiche §5) ?

In [ ]:
def height_21(p):
    """y(v, θ, t) = v t sin θ − 5 t², in m: the height of the projectile of the fiche, for p = [v, θ, t] (duals)."""
    v, theta, t = p
    return v * t * theta.sin() - 5 * t ** 2


def range_21(p):
    """R(v, θ) = v² sin(2θ) / 10, in m: where a ball thrown from the ground falls back, for p = [v, θ] (duals)."""
    v, theta = p
    return v ** 2 * (2 * theta).sin() / 10


POINT_HEIGHT_21 = [20.0, math.pi / 6, 0.5]       # (v, θ, t) of the fiche
POINT_RANGE_21 = [20.0, math.pi / 8]             # (v, θ)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M18")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M18, "forward_gradient")

**L'usage** : les gradients de la hauteur et de la portée, et ce qu'ils coûtent.

In [ ]:
answer_M18_21a = answer_M18_21b = answer_M18_21c = ...
with wbm.attempt("M18.21"):          # if you call forward_gradient here: ⏳ while it is not written
    answer_M18_21a = ...  # a) a list of 3 numbers (4 decimals): the gradient of the height at POINT_HEIGHT_21
    answer_M18_21b = ...  # b) a list of 2 numbers (4 decimals): the gradient of the range at POINT_RANGE_21
    answer_M18_21c = ...  # c) a list of 2 integers: the calls of the function by forward_gradient, for a) then b)

In [ ]:
for letter, answer in zip("abc", [answer_M18_21a, answer_M18_21b, answer_M18_21c]):
    wbm.check(f"M18.21{letter}", answer, computed=True)

### Ex M18.22 — Lire le gradient dans la différentielle : G[i, j] = df(E_ij), de la forme de la variable · `gradient_from_differential` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer la lecture du gradient d'une fonction de matrice dans sa différentielle, case par case, et retrouver la comatrice comme gradient du déterminant.  
**Prérequis :** fiche §6 et §7 · **mymath :** `autodiff.py` · **Parcours :** C, E

Écris `gradient_from_differential(differential, shape)` (fiche §6 et §7), en suivant sa docstring. Près de $\mathbf{X}$, une fonction réelle d'une matrice bouge de $\mathrm{d}f = \sum_{ij}G_{ij}H_{ij} = \operatorname{tr}(\mathbf{G}^\top\mathbf{H})$ quand $\mathbf{X}$ bouge de $\mathbf{H}$ : la différentielle est une fonction **linéaire** du déplacement, et le gradient $\mathbf{G}$ est le tableau de ses coefficients. Sa case $(i, j)$ est la valeur de la différentielle sur le déplacement $\mathbf{E}_{ij}$, qui a un 1 en $(i, j)$ et des 0 ailleurs : $G_{ij} = \mathrm{d}f(\mathbf{E}_{ij})$. Ta fonction rend donc un tableau de flottants **de la forme de la variable** ; `shape` est un entier (pour un vecteur) ou un tuple d'entiers, tous $> 0$ (`ValueError` sinon), et la même lecture vaut pour un tableau à trois indices (`np.ndindex(shape)` parcourt toutes les cases). Elle ne vérifie pas que la différentielle est linéaire.

**L'usage : le gradient du déterminant.** Pour une matrice carrée inversible, $\mathrm{d}(\det\mathbf{X}) = \det\mathbf{X}\,\operatorname{tr}(\mathbf{X}^{-1}\mathbf{H})$ (fiche §7). La cellule fournie donne une matrice $3 \times 3$ tirée au hasard (graine 5131, un chiffre après la virgule ; `X_22`) et `comatrix_22`, sa comatrice calculée par les cofacteurs : $C_{ij}$ est $(-1)^{i+j}$ fois le déterminant de la matrice privée de sa ligne $i$ et de sa colonne $j$. Écris la différentielle comme une fonction de `H` (`np.linalg.det`, et `np.linalg.solve(X_22, H)` pour $\mathbf{X}^{-1}\mathbf{H}$), enveloppe-la dans `Counted`, puis, avec ta fonction, calcule :
- a) le gradient de $\det$ en `X_22` (une matrice, 4 décimales) ;
- b) vrai ou faux : il coïncide avec `comatrix_22(X_22)` à $10^{-9}$ près (la plus grande différence, en valeur absolue) ;
- c) le nombre d'appels de la différentielle (un entier).

Dans ta copie : pourquoi le gradient du déterminant est-il la comatrice (fiche §7, ∂ M18.14) ? Vérifie que $\sum_{ij}G_{ij}X_{ij} = 3\det\mathbf{X}$, et explique-le par le développement du déterminant selon chacune des trois lignes. Combien d'appels faudrait-il pour une matrice $100 \times 100$ ? Ta fonction a-t-elle eu besoin de $f$ elle-même, ou seulement de sa différentielle ?

In [ ]:
X_22 = np.round(np.random.default_rng(5131).normal(size=(3, 3)), 1)    # a 3 × 3 matrix, one decimal per entry


def comatrix_22(X):
    """The matrix of the cofactors of X: C[i, j] = (−1)^(i+j) det(X without its row i and its column j)."""
    X = np.asarray(X, dtype=float)
    n = len(X)
    C = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            minor = np.delete(np.delete(X, i, axis=0), j, axis=1)
            C[i, j] = (-1) ** (i + j) * np.linalg.det(minor)
    return C

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M18")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M18, "gradient_from_differential")

**L'usage** : le gradient du déterminant, lu dans sa différentielle, contre la comatrice.

In [ ]:
answer_M18_22a = answer_M18_22b = answer_M18_22c = ...
with wbm.attempt("M18.22"):          # if you call gradient_from_differential here: ⏳ while it is not written
    answer_M18_22a = ...  # a) a matrix (4 decimals): the gradient of det at X_22, read in its differential
    answer_M18_22b = ...  # b) True or False
    answer_M18_22c = ...  # c) an integer: the calls of the differential

In [ ]:
for letter, answer in zip("abc", [answer_M18_22a, answer_M18_22b, answer_M18_22c]):
    wbm.check(f"M18.22{letter}", answer, computed=True)

## Partie B · Comparer, prédire, mesurer

Fiche §3, §5, §7 et §8. SymPy contre les identités du calcul matriciel ; le coût des deux modes, prédit puis compté ; l'erreur des différences centrées contre la dérivée duale.

### Ex M18.23 — SymPy : dériver une fonction de matrice symbolique, et comparer aux identités 📦 ★ ⏱️ 25 min
**Objectif :** dériver des fonctions de matrices avec `sympy.MatrixSymbol`, comparer aux identités de la fiche, et repérer un cas où SymPy se trompe sans prévenir.  
**Prérequis :** fiche §7 · **Parcours :** C

SymPy dérive aussi des expressions de **matrices symboliques** : `X = sympy.MatrixSymbol("X", 2, 2)` est une matrice $2 \times 2$ dont on ne connaît pas les coefficients, et `expr.diff(X)` rend le gradient par rapport à `X`, de la forme de `X` (la convention de la fiche). Avec `A = sympy.MatrixSymbol("A", 2, 2)`, et `a`, `b` de forme $(2, 1)$ :
- `sympy.trace(A * X).diff(X)`, `sympy.trace(X.T * A * X).diff(X)` et `(a.T * X * b).diff(X)` rendent des expressions de matrices ;
- `expr.subs({A: sympy.Matrix(A_23), X: sympy.Matrix(X_23), a: sympy.Matrix(a_23), b: sympy.Matrix(b_23)}).doit()` les évalue en des matrices de nombres, et `np.array(..., dtype=float)` en fait des tableaux NumPy.

⚠️ Pour $\ln\det\mathbf{X}$, essaie `sympy.log(sympy.Determinant(X)).diff(X)` : SymPy (1.13 et 1.14) rend `0`, sans prévenir, et c'est faux. Passe alors par une matrice de **symboles**, `sympy.Matrix(2, 2, sympy.symbols("x11 x12 x21 x22"))`, dont `.det()` est un polynôme : dérive $\ln\det$ par rapport à chacun des quatre symboles, range les dérivées comme la matrice, puis remplace les symboles par les coefficients de `X_23`.

La cellule fournie donne $\mathbf{A}$ (`A_23`), $\mathbf{X}$ (`X_23`, qui n'est pas symétrique et dont le déterminant est positif), $\mathbf{a}$ et $\mathbf{b}$ (`a_23`, `b_23`). Calcule avec SymPy :
- a) le gradient de $\operatorname{tr}(\mathbf{X}^\top\mathbf{A}\mathbf{X})$ par `MatrixSymbol`, évalué en `A_23` et `X_23` (une matrice) ;
- b) vrai ou faux : les trois gradients de SymPy, évalués en `A_23`, `X_23`, `a_23` et `b_23`, coïncident avec $\mathbf{A}^\top$, $(\mathbf{A} + \mathbf{A}^\top)\mathbf{X}$ et $\mathbf{a}\mathbf{b}^\top$ (fiche §7 ; `np.allclose`) ;
- c) le gradient de $\ln\det\mathbf{X}$ en `X_23`, par la matrice de symboles (une matrice, 4 décimales).

Dans ta copie : écris les trois expressions que rend SymPy et compare-les aux identités de la fiche. Compare c) à $\mathbf{X}^{-1}$ et à $\mathbf{X}^{-\top}$ : lequel est-ce, et pourquoi la différence se voit-elle ici ? Pourquoi la réponse `0` de SymPy pour $\ln\det$ est-elle dangereuse, et comment une vérification le long d'une direction (fiche §8) l'aurait-elle démasquée sans connaître la formule ?

In [ ]:
A_23 = np.array([[0.0, 2.0],
                 [-3.0, 1.0]])
X_23 = np.array([[3.0, 1.0],
                 [-2.0, 2.0]])                  # not symmetric; det X_23 = 8 > 0
a_23 = np.array([1.0, 2.0])
b_23 = np.array([-1.0, 4.0])

In [ ]:
answer_M18_23a = ...  # a) a matrix: the gradient of tr(Xᵀ A X) by MatrixSymbol, at A_23 and X_23
answer_M18_23b = ...  # b) True or False
answer_M18_23c = ...  # c) a matrix (4 decimals): the gradient of ln det X at X_23, by a matrix of symbols

In [ ]:
for letter, answer in zip("abc", [answer_M18_23a, answer_M18_23b, answer_M18_23c]):
    wbm.check(f"M18.23{letter}", answer)

### Ex M18.24 — Prédire le nombre d'opérations de chaque mode, puis le compter 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir ce que coûte le gradient d'un produit de $n$ nombres en mode direct et en mode inverse, puis le compter, et voir comment chaque coût grandit avec $n$.  
**Prérequis :** fiche §5 ; Ex M18.20 · **Parcours :** C, E

On dérive $f(\mathbf{v}) = v_1v_2\cdots v_n$, le produit de $n$ nombres calculé une multiplication après l'autre (`product_24`), en $n = 8$ nombres (`VALUES_24`), de deux façons :
- en **mode direct**, avec **ta** classe `Dual` : une passe par entrée, comme `forward_gradient` (la passe $i$ évalue $f$ sur les duaux `Dual(v_j, 1.0 if j == i else 0.0)`) ;
- en **mode inverse**, par une passe avant puis une passe arrière **écrites à la main** (`product_by_hand_24`, cellule fournie) : $u_1 = v_1$, $u_k = u_{k-1}v_k$, $f = u_n$, puis, de $k = n$ à 2, $\bar{v}_k = \bar{u}_k\,u_{k-1}$ et $\bar{u}_{k-1} = \bar{u}_k\,v_k$, et enfin $\bar{v}_1 = \bar{u}_1$ ; son compteur ne compte que les produits « adjoint × dérivée locale » de la passe arrière, comme en M18.5 g).

L'expérience compte, pour $n = 8$ puis $n = 16$, les évaluations de $f$ et les multiplications de duals du mode direct (les appels de `Dual.__mul__` et `Dual.__rmul__` de ta classe, enveloppés le temps des passes), et les produits de la passe arrière à la main ; elle compare aussi les deux gradients. Pour chaque question, écris un entier (le numéro du choix pour d) :
- a) pour $n = 8$, le nombre d'évaluations de $f$ du mode direct, pour tout le gradient ;
- b) pour $n = 8$, le nombre total de multiplications de duals, sur toutes les passes ;
- c) pour $n = 8$, le nombre de produits « adjoint × dérivée locale » de la passe arrière de `product_by_hand_24` ;
- d) quand $n$ passe de 8 à 16 : (1) les deux comptes, b) et c), doublent environ ; (2) celui du mode direct est multiplié par 4 environ, celui de la passe arrière par 2 environ ; (3) les deux sont multipliés par 4 environ ; (4) celui du mode direct double environ, celui de la passe arrière est multiplié par 4 environ.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions, vérifie-les, puis lance l'expérience.

In [ ]:
VALUES_24 = [1.5, -0.5, 2.0, 0.8, -1.25, 3.0, 0.4, 2.5]       # the 8 numbers v_1, ..., v_8


def product_24(v):
    """f(v) = v_1 v_2 ... v_n, one multiplication after the other (numbers or duals)."""
    result = v[0]
    for factor in v[1:]:
        result = result * factor
    return result


def product_by_hand_24(v, counter):
    """f(v) = v_1 ... v_n and its gradient, by a forward pass then a backward pass written by hand:
    u_1 = v_1, u_k = u_(k-1) v_k, f = u_n. counter[0] counts the products « adjoint × local derivative » of the
    backward pass."""
    n = len(v)
    u = [v[0]]                                   # forward pass: keep every u_k (not counted)
    for k in range(1, n):
        u.append(u[k - 1] * v[k])
    v_bar = [0.0] * n
    u_bar = 1.0                                  # the adjoint of f = u_n
    for k in range(n - 1, 0, -1):                # backward pass, u_k = u_(k-1) · v_k
        v_bar[k] = u_bar * u[k - 1]              # ∂u_k/∂v_k = u_(k-1)
        u_bar = u_bar * v[k]                     # ∂u_k/∂u_(k-1) = v_k: now the adjoint of u_(k-1)
        counter[0] += 2
    v_bar[0] = u_bar                             # u_1 = v_1
    return u[-1], v_bar


def forward_mode_count_24(Dual, values):
    """The gradient of product_24 at values by n forward passes with the class Dual, the number of evaluations of f,
    and the number of multiplications of duals (calls of Dual.__mul__ and Dual.__rmul__, wrapped during the passes;
    a call made inside another one is not counted again)."""
    calls, depth = [0], [0]
    originals = {name: vars(Dual)[name] for name in ("__mul__", "__rmul__") if name in vars(Dual)}

    def counting(method):
        def wrapper(self, other):
            calls[0] += depth[0] == 0
            depth[0] += 1
            try:
                return method(self, other)
            finally:
                depth[0] -= 1
        return wrapper

    for name, method in originals.items():
        setattr(Dual, name, counting(method))
    try:
        gradient, evaluations = [], 0
        for i in range(len(values)):
            y = product_24([Dual(x, 1.0 if j == i else 0.0) for j, x in enumerate(values)])
            evaluations += 1
            gradient.append(y.derivative)
    finally:
        for name, method in originals.items():
            setattr(Dual, name, method)
    return gradient, evaluations, calls[0]

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M18_24a = ...  # a) an integer, written BEFORE running the experiment below
prediction_M18_24b = ...  # b) an integer
prediction_M18_24c = ...  # c) an integer
prediction_M18_24d = ...  # d) 1, 2, 3 or 4

In [ ]:
for letter, prediction in zip("abcd", [prediction_M18_24a, prediction_M18_24b, prediction_M18_24c,
                                         prediction_M18_24d]):
    wbm.check(f"M18.24{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M18_24a, prediction_M18_24b, prediction_M18_24c, prediction_M18_24d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    with checked("M18.24"):
        for n_24 in (8, 16):
            values_n_24 = (VALUES_24 * 2)[:n_24]
            gradient_24, passes_24, products_24 = forward_mode_count_24(mymath.autodiff.Dual, values_n_24)
            counter_24 = [0]
            _, by_hand_24 = product_by_hand_24(values_n_24, counter_24)
            print(f"n = {n_24} : mode direct, {passes_24} évaluations de f et {products_24} multiplications de duals ; "
                  f"passe arrière à la main, {counter_24[0]} produits « adjoint × dérivée locale » ; mêmes gradients : "
                  f"{bool(np.allclose(gradient_24, by_hand_24))}")

**Après l'expérience**, dans ta copie : écris les deux comptes pour $n$ quelconque, et dis comment chacun grandit avec $n$. D'où vient le carré du mode direct (combien de passes, et combien de multiplications par passe) ? Combien de fois la passe arrière passe-t-elle sur chaque arête ? Que doit garder en mémoire la passe arrière, et le mode direct ? Pour une fonction de $\mathbb{R}$ dans $\mathbb{R}^{1\,000}$, lequel des deux modes choisirais-tu (fiche §5) ?

### Ex M18.25 — Différences finies contre nombres duaux : l'erreur selon le pas h 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer l'erreur d'une différence centrée selon le pas $h$, y lire la troncature et les arrondis, et la comparer à la dérivée duale, sans pas.  
**Prérequis :** fiche §3 et §8 ; M05a ; Ex M18.20 · **Parcours :** C

Une dérivée par **différence centrée** (M05a) dépend d'un pas $h$ ; la dérivée par **nombres duaux** n'en a pas (fiche §3). On les compare sur $f(x) = e^{\sin x}$ en $x = 1$ (cellule fournie : `f_25` pour un flottant, `f_dual_25` pour un `Dual`, et les treize pas $h = 10^{-1}, 10^{-2}, \ldots, 10^{-13}$ de `STEPS_25`).
- Calcule la dérivée exacte $f'(1)$ avec SymPy (`sympy.diff`, puis `.subs` et `float`) ;
- range dans `errors_25` les treize erreurs $\lvert D_h - f'(1)\rvert$, où $D_h = \frac{f(1 + h) - f(1 - h)}{2h}$, dans l'ordre de `STEPS_25` ;
- la cellule de vérification trace ces erreurs en fonction de $h$, les deux axes en échelle logarithmique, avec l'écart de la dérivée duale.

Réponds :
- a) l'entier $k$ du meilleur pas, $h = 10^{-k}$ ;
- b) l'entier $m$ tel que la meilleure erreur soit comprise entre $10^m$ et $10^{m+1}$ ;
- c) sur la figure, la pente de la courbe à droite (les grands pas), arrondie à l'entier ;
- d) la pente de la courbe à gauche (les petits pas), arrondie à l'entier ;
- e) vrai ou faux : la dérivée de `dual_derivative` s'écarte de $f'(1)$ de moins de $10^{-14}$.

Dans ta copie : d'où viennent les deux pentes (l'erreur de troncature de la différence centrée, M05a, et les arrondis de $f(1 \pm h)$, de l'ordre de $10^{-16}$, divisés par $2h$) ? Écris l'erreur totale comme la somme de deux termes en $h$, et retrouve l'ordre de grandeur du meilleur pas et de la meilleure erreur. Pourquoi la dérivée duale n'a-t-elle ni l'une ni l'autre de ces erreurs ? Pour quelles fonctions ne peut-on pas s'en servir (pense à une fonction calculée par un programme dont tu n'as pas le code), et que reste-t-il alors ?

In [ ]:
STEPS_25 = [10.0 ** -k for k in range(1, 14)]     # h = 1e-1, 1e-2, ..., 1e-13
X0_25 = 1.0


def f_25(x):
    """f(x) = e^(sin x), for a float."""
    return math.exp(math.sin(x))


def f_dual_25(t):
    """The same f for a Dual, written with its methods."""
    return t.sin().exp()

In [ ]:
x_25 = sympy.Symbol("x")
exact_25 = ...        # f'(1) by SymPy, as a float
errors_25 = ...       # the 13 errors |D_h − f'(1)|, in the order of STEPS_25
answer_M18_25a = ...  # a) an integer k: the best step is h = 10^(-k)
answer_M18_25b = ...  # b) an integer m: the best error lies between 10^m and 10^(m+1)
answer_M18_25c = ...  # c) an integer: the slope of the curve on the right (large steps), read on the figure
answer_M18_25d = ...  # d) an integer: the slope on the left (small steps)
answer_M18_25e = ...
with wbm.attempt("M18.25"):          # dual_derivative: ⏳ while it is not written
    answer_M18_25e = ...  # e) True or False

In [ ]:
with checked("M18.25 figure"):
    if filled(errors_25):
        fig, ax = plt.subplots(figsize=(7.0, 4.2))
        try:
            ax.loglog(STEPS_25, np.maximum(np.abs(np.asarray(errors_25, dtype=float)), 1e-18), "o-", color="C0",
                      label="différence centrée")
            with wbm.attempt("M18.25"):          # the dual derivative: ⏳ while dual_derivative is not written
                if filled(exact_25):
                    gap_25 = abs(mymath.autodiff.dual_derivative(f_dual_25, X0_25)[1] - float(exact_25))
                    ax.axhline(max(gap_25, 1e-17), color="C3", ls="--", label=f"nombres duaux (écart {gap_25:.0e})")
            ax.set_xlabel("pas h (échelle log)")
            ax.set_ylabel("erreur |D_h − f′(1)| (échelle log)")
            ax.grid(True, which="both", alpha=0.3)
            ax.legend(fontsize=8)
        except BaseException:
            plt.close(fig)
            raise
        plt.show()
    else:
        print("⏳ Ex M18.25 : la figure attend tes treize erreurs, errors_25.")
for letter, answer in zip("abcd", [answer_M18_25a, answer_M18_25b, answer_M18_25c, answer_M18_25d]):
    wbm.check(f"M18.25{letter}", answer)
wbm.check("M18.25e", answer_M18_25e, computed=True)

## Partie C · Dessiner, déboguer, tester, défier

Fiche §1, §3, §4, §6 et §8. Le graphe de calcul d'une formule, avec ses dérivées locales et ses adjoints ; trois erreurs classiques d'un gradient et de sa vérification ; des tests de propriété pour une classe de nombres duaux ; un défi : dériver la méthode de Newton. Les vérifications de M18.26 et de M18.28 s'appuient sur la référence (`ref_autodiff`), pour ne pas dépendre de ton code.

In [ ]:
ref_autodiff = wbm.load_mymath("ref", alias="mymath_ref").autodiff   # the reference: the checks of M18.26 and M18.28

### Ex M18.26 — Dessiner un graphe de calcul, avec les dérivées locales sur les arêtes et les adjoints sur les nœuds 🎨 ★ ⏱️ 30 min
**Objectif :** dessiner le graphe de calcul d'une formule avec les valeurs de la passe avant, les dérivées locales des arêtes et les adjoints de la passe arrière.  
**Prérequis :** fiche §1, §2 et §4 · **Parcours :** C

Dessine le graphe de calcul de la résistance équivalente de deux résistances montées en parallèle (M18.Q1), $R = \frac{r_1r_2}{r_1 + r_2}$, calculée par $u_1 = r_1r_2$, $u_2 = r_1 + r_2$, $R = \frac{u_1}{u_2}$, en $(r_1, r_2) = (3, 6)$ (en ohms) : les valeurs de la passe avant dans les nœuds, la **dérivée locale** $\frac{\partial w}{\partial u}$ sur chaque arête $u \to w$, et l'**adjoint** $\bar{u} = \frac{\partial R}{\partial u}$ à côté de chaque nœud (fiche §1, §2 et §4 ; comme la figure du §1).
1. Calcule d'abord, à la main, puis range dans les trois dictionnaires de la cellule à compléter : les 5 valeurs (`VALUES_26`), les 6 dérivées locales (`LOCAL_26`, une par arête de `EDGES_26`) et les 5 adjoints (`ADJOINTS_26`, en partant de $\bar{R} = 1$, dans l'ordre inverse du calcul). Des nombres, ou des calculs comme `4 / 9`.
2. Écris `draw_graph(ax, positions, values, local, adjoints)`, qui dessine dans l'axe `ax` (sans en ouvrir d'autre) :
   - chaque nœud à sa position (`positions`, ici `POSITIONS_26`), avec son nom et sa valeur : `ax.text(x, y, ..., ha="center", va="center", bbox=dict(boxstyle="round", facecolor="white"))` ;
   - chaque arête $u \to w$ comme une flèche **de $u$ vers $w$** : `ax.annotate("", xy=positions[w], xytext=positions[u], arrowprops=dict(arrowstyle="->", shrinkA=22, shrinkB=22))` (la pointe en `xy`, le départ en `xytext` ; `shrinkA` et `shrinkB` raccourcissent la flèche, en points, pour qu'elle ne cache pas les nœuds) ;
   - la dérivée locale de chaque arête à la position `EDGE_LABELS_26[(u, w)]`, et l'adjoint de chaque nœud à la position `ADJOINT_LABELS_26[node]` (cellule fournie) : les nombres en clair, `0.444` ou `4/9` ;
   - des axes cachés (`ax.set_axis_off()`) et des limites qui montrent tout (`ax.set_xlim`, `ax.set_ylim`).

La vérification compare tes nombres à ceux de la référence (chacun est une dérivée, qu'elle calcule avec ses nombres duaux), puis appelle ta fonction et regarde le dessin : les six flèches, dans le bon sens, et un texte qui porte le bon nombre à chacune des positions données.

In [ ]:
NODES_26 = ["r1", "r2", "u1", "u2", "R"]
EDGES_26 = [("r1", "u1"), ("r2", "u1"), ("r1", "u2"), ("r2", "u2"), ("u1", "R"), ("u2", "R")]   # u → w: w uses u
POSITIONS_26 = {"r1": (0.0, 1.0), "r2": (0.0, -1.0), "u1": (2.5, 1.0), "u2": (2.5, -1.0), "R": (5.0, 0.0)}
EDGE_LABELS_26 = {("r1", "u1"): (1.25, 1.22), ("r2", "u1"): (0.75, -0.4), ("r1", "u2"): (0.75, 0.4),
                  ("r2", "u2"): (1.25, -1.22), ("u1", "R"): (3.75, 0.74), ("u2", "R"): (3.75, -0.74)}
ADJOINT_LABELS_26 = {"r1": (0.0, 1.5), "r2": (0.0, -1.5), "u1": (2.5, 1.5), "u2": (2.5, -1.5), "R": (5.0, -0.5)}

In [ ]:
VALUES_26 = {"r1": ..., "r2": ..., "u1": ..., "u2": ..., "R": ...}        # the values of the forward pass
LOCAL_26 = {("r1", "u1"): ..., ("r2", "u1"): ..., ("r1", "u2"): ..., ("r2", "u2"): ...,
            ("u1", "R"): ..., ("u2", "R"): ...}                            # the local derivative ∂w/∂u of each edge
ADJOINTS_26 = {"r1": ..., "r2": ..., "u1": ..., "u2": ..., "R": ...}      # the adjoints ∂R/∂u


def draw_graph(ax, positions, values, local, adjoints):
    """In ax (and no other axes): each node at its position with its name and value, each edge u → w as an arrow
    (ax.annotate) with its local derivative at EDGE_LABELS_26[(u, w)], each adjoint at ADJOINT_LABELS_26[node]."""
    raise NotImplementedError("draw_graph n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M18.26"):
    results_26 = []

    def report_26(ok, success, failure):
        results_26.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    if not filled(*VALUES_26.values(), *LOCAL_26.values(), *ADJOINTS_26.values()):
        print("⏳ Ex M18.26 : pas encore fait (remplis les trois dictionnaires).")
        raise StopCheck
    dd_26 = ref_autodiff.dual_derivative                      # each number as a derivative, by the reference
    r1_26, r2_26 = 3.0, 6.0
    u1_26, u2_26 = r1_26 * r2_26, r1_26 + r2_26
    want_values_26 = [r1_26, r2_26, u1_26, u2_26, u1_26 / u2_26]
    want_local_26 = [dd_26(lambda t: t * r2_26, r1_26)[1], dd_26(lambda t: r1_26 * t, r2_26)[1],
                     dd_26(lambda t: t + r2_26, r1_26)[1], dd_26(lambda t: r1_26 + t, r2_26)[1],
                     dd_26(lambda t: t / u2_26, u1_26)[1], dd_26(lambda t: u1_26 / t, u2_26)[1]]
    want_adjoints_26 = [dd_26(lambda t: t * r2_26 / (t + r2_26), r1_26)[1],
                        dd_26(lambda t: r1_26 * t / (r1_26 + t), r2_26)[1],
                        dd_26(lambda t: t / u2_26, u1_26)[1], dd_26(lambda t: u1_26 / t, u2_26)[1], 1.0]
    results_26.append(same("M18.26", "les valeurs [r1, r2, u1, u2, R]", [VALUES_26[k] for k in NODES_26],
                           want_values_26))
    results_26.append(same("M18.26", "les dérivées locales, dans l'ordre de EDGES_26",
                           [LOCAL_26[e] for e in EDGES_26], want_local_26))
    results_26.append(same("M18.26", "les adjoints [r1, r2, u1, u2, R]", [ADJOINTS_26[k] for k in NODES_26],
                           want_adjoints_26))

    def node_at_26(point):
        """The node whose position is within 0.6 of point, or None."""
        best = min(NODES_26, key=lambda k: math.dist(POSITIONS_26[k], point))
        return best if math.dist(POSITIONS_26[best], point) < 0.6 else None

    def numbers_26(text):
        """The numbers written in a text: 18, 0.444, 0,444, 4/9, −2/9, \\frac{4}{9}."""
        text = text.replace("−", "-").replace(",", ".")
        text = re.sub(r"\\d?frac\{(-?\d+)\}\{(\d+)\}", r"\1/\2", text)
        return [float(a) / float(b) if b else float(a)
                for a, b in re.findall(r"(-?\d+(?:\.\d+)?)(?:\s*/\s*(\d+(?:\.\d+)?))?", text)
                if not b or float(b) != 0]

    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    try:
        opened_26 = drawn(draw_graph, ax, dict(POSITIONS_26), dict(VALUES_26), dict(LOCAL_26), dict(ADJOINTS_26))
        report_26(not opened_26, "draw_graph dessine dans l'axe reçu",
                  "draw_graph doit dessiner dans l'axe reçu, sans ouvrir de figure (ni plt.figure ni plt.subplots)")
        arrows_26, texts_26 = set(), []
        for item in ax.texts:
            if isinstance(item, Annotation):
                if item.arrow_patch is not None and item.xycoords == "data" and item.anncoords == "data":
                    arrows_26.add((node_at_26(item.xyann), node_at_26(item.xy)))       # (start, head)
                if item.get_text().strip() and item.anncoords == "data":
                    texts_26.append((tuple(item.xyann), item.get_text()))
            elif item.get_text().strip() and item.get_transform() == ax.transData:
                texts_26.append((tuple(item.get_position()), item.get_text()))
        missing_26 = [f"{u} → {w}" for u, w in EDGES_26 if (u, w) not in arrows_26]
        backwards_26 = [f"{u} → {w}" for u, w in EDGES_26 if (u, w) not in arrows_26 and (w, u) in arrows_26]
        report_26(not missing_26, "les six flèches vont de u vers w",
                  f"il manque la flèche de {len(missing_26)} arête(s) : {', '.join(missing_26)}"
                  + (f" (dans le mauvais sens : {', '.join(backwards_26)} ; la pointe est en xy, le départ en "
                     f"xytext)" if backwards_26 else " (ax.annotate, la pointe en xy, le départ en xytext, en "
                     "coordonnées des données)"))

        def shown_26(point, value):
            near = [text for where, text in texts_26 if math.dist(where, point) < 0.3]
            return any(abs(x - value) <= max(0.01, 0.01 * abs(value)) for text in near for x in numbers_26(text))

        for kind_26, places_26, wanted_26 in (
                ("les valeurs des nœuds", [POSITIONS_26[k] for k in NODES_26], want_values_26),
                ("les dérivées locales", [EDGE_LABELS_26[e] for e in EDGES_26], want_local_26),
                ("les adjoints", [ADJOINT_LABELS_26[k] for k in NODES_26], want_adjoints_26)):
            absent_26 = [f"{tuple(round(c, 2) for c in p)}" for p, v in zip(places_26, wanted_26) if not shown_26(p, v)]
            report_26(not absent_26, f"{kind_26} sont à leur place",
                      f"{kind_26} : aucun texte qui porte le bon nombre, en clair (0.444 ou 4/9), aux positions "
                      f"{', '.join(absent_26[:3])}" + (" …" if len(absent_26) > 3 else ""))
    except BaseException:
        plt.close(fig)
        raise
    plt.show()
    verdict("M18.26", all(results_26), "les nombres sont justes et le graphe est dessiné : nœuds, flèches, dérivées "
                                       "locales et adjoints.", "relis les lignes ❌ ci-dessus.")

**Regarde ta figure**, puis, dans ta copie : quels nœuds reçoivent deux contributions dans la passe arrière, et lesquelles ? Retrouve $\bar{r}_1$ et $\bar{r}_2$ en dérivant $R$ directement. Que vaut $3\bar{r}_1 + 6\bar{r}_2$, et pourquoi (que devient $R$ quand on multiplie $r_1$ et $r_2$ par un même nombre) ? Laquelle des deux résistances faut-il changer pour faire bouger $R$ le plus ?

### Ex M18.27 — Chasse à l'erreur : un gradient transposé, un adjoint compté sur un seul chemin, un pas h trop petit 🐛 ★ ⏱️ 15 min
**Objectif :** reconnaître trois erreurs classiques du calcul et de la vérification d'un gradient qui ne lèvent aucune erreur Python, puis les corriger.  
**Prérequis :** fiche §4, §6 et §8 · **Parcours :** C, E

Un collègue a écrit trois morceaux de code (cellule fournie) ; chacun a l'erreur que nomme le titre, et aucun ne lève d'erreur Python :
1. `grad_trace(A, X)` doit rendre le gradient de $\mathbf{X} \mapsto \operatorname{tr}(\mathbf{A}\mathbf{X})$, de la forme de $\mathbf{X}$. Pour le vérifier, il a écrit `check_along(f, G, X, H, h)`, qui compare la différence centrée de $f$ le long d'une direction $\mathbf{H}$, $\frac{f(\mathbf{X} + h\mathbf{H}) - f(\mathbf{X} - h\mathbf{H})}{2h}$, à la prévision $\sum_{ij}G_{ij}H_{ij}$ (fiche §8), et rend `True` quand elles diffèrent de moins de $10^{-6}$. Il l'a appelée en `A_27` et `X_27`, le long de la direction $\mathbf{I}$ : son gradient est passé.
2. `adjoints_27(x, y)` rend la valeur de $f(x, y) = x\sin(xy)$ et son gradient $(\frac{\partial f}{\partial x}, \frac{\partial f}{\partial y})$, par une passe avant ($u_1 = xy$, $u_2 = \sin u_1$, $f = x\,u_2$) puis une passe arrière écrites ligne à ligne ; l'un de ses adjoints est **incomplet** : il lui manque la contribution d'un chemin.
3. Il a vérifié la formule du gradient de $f$ (`gradient_27`, qui est juste) avec `check_along` et $h = 10^{-13}$, le long de la direction $(1, -1)$ en $(1{,}5 ;\ -0{,}5)$ (`DIRECTION_27`, `POINT_27`) : sa vérification dit `False`, et il croit la formule fausse.

À faire :
- a) d'abord, à la main : le $\bar{x}$ que rend `adjoints_27(1.5, -0.5)` (4 décimales) ; vérifie-le en appelant la fonction ;
- (1) tire une direction `H_27` de la forme de `X_27`, avec `np.random.default_rng(5137)` (des coefficients de loi normale), vérifie qu'elle démasque `grad_trace`, puis écris `grad_trace_fixed(A, X)` ;
- (2) écris `adjoints_27_fixed(x, y)`, avec la même passe avant et une passe arrière juste ; b) le gradient $(\bar{x}, \bar{y})$ qu'elle rend en $(1{,}5 ;\ -0{,}5)$ (une liste de 2 nombres, 4 décimales) ;
- (3) choisis le pas `STEP_27` d'une vérification fiable (fiche §8 ; M05a).

La vérification essaie tes corrections sur des cas que tu ne vois pas : d'autres matrices, carrées ou non, d'autres points, d'autres directions.

In [ ]:
A_27 = np.array([[2.0, 3.0],
                 [-1.5, 1.0]])
X_27 = np.array([[1.5, 0.0],
                 [-1.0, 2.5]])
POINT_27 = np.array([1.5, -0.5])               # the point (x, y) of (2) and (3)
DIRECTION_27 = np.array([1.0, -1.0])           # the direction of the colleague's check in (3)


def check_along(f, G, X, H, h=1e-5):
    """Compare the central difference of f along the direction H with the prediction Σ G ⊙ H (fiche §8). Returns
    (True when they differ by less than 1e-6, the central difference, the prediction)."""
    X, H = np.asarray(X, dtype=float), np.asarray(H, dtype=float)
    numerical = (f(X + h * H) - f(X - h * H)) / (2 * h)
    predicted = float(np.sum(np.asarray(G, dtype=float) * H))
    return bool(abs(numerical - predicted) < 1e-6), float(numerical), predicted


def trace_27(X):
    """f(X) = tr(A_27 X)."""
    return float(np.trace(A_27 @ X))


def f_27(p):
    """f(x, y) = x sin(x y), for p = (x, y)."""
    return p[0] * math.sin(p[0] * p[1])


def gradient_27(p):
    """The gradient of f_27 at p by its formula, (sin(xy) + xy cos(xy), x² cos(xy)): right (checked)."""
    x, y = p
    return np.array([math.sin(x * y) + x * y * math.cos(x * y), x ** 2 * math.cos(x * y)])


# The colleague's code
def grad_trace(A, X):
    """The gradient of X ↦ tr(A X), of the shape of X."""
    return A


def adjoints_27(x, y):
    """f(x, y) = x sin(x y) and its gradient (∂f/∂x, ∂f/∂y), by a forward pass then a backward pass, line by line."""
    u1 = x * y                       # forward pass
    u2 = math.sin(u1)
    f = x * u2
    f_bar = 1.0                      # backward pass, in the reverse order
    u2_bar = f_bar * x               # f = x · u2: ∂f/∂u2 = x
    u1_bar = u2_bar * math.cos(u1)   # u2 = sin u1
    x_bar = u1_bar * y               # u1 = x · y: ∂u1/∂x = y
    y_bar = u1_bar * x               # ∂u1/∂y = x
    return f, x_bar, y_bar


print("(1) check_along de grad_trace le long de I :", check_along(trace_27, grad_trace(A_27, X_27), X_27, np.eye(2)))
print("(3) check_along de gradient_27 avec h = 1e-13 :",
      check_along(f_27, gradient_27(POINT_27), POINT_27, DIRECTION_27, h=1e-13))

In [ ]:
answer_M18_27a = ...  # a) a number (4 decimals): the x̄ that the colleague's adjoints_27 returns at (1.5, -0.5)

H_27 = ...            # (1) a direction of the shape of X_27, drawn with np.random.default_rng(5137)


def grad_trace_fixed(A, X):
    """The gradient of X ↦ tr(A X), of the shape of X."""
    raise NotImplementedError("grad_trace_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def adjoints_27_fixed(x, y):
    """f(x, y) = x sin(x y) and its gradient (∂f/∂x, ∂f/∂y), by a forward pass then a backward pass, line by line."""
    raise NotImplementedError("adjoints_27_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


STEP_27 = ...         # (3) the step h of a reliable check by check_along
answer_M18_27b = ...
with wbm.attempt("M18.27"):          # adjoints_27_fixed: ⏳ while it is not written
    answer_M18_27b = ...  # b) a list of 2 numbers (4 decimals): the gradient that adjoints_27_fixed returns

In [ ]:
wbm.check("M18.27a", answer_M18_27a)
wbm.check("M18.27b", answer_M18_27b, computed=True)

results_27, parts_27 = [], []


def report_27(ok, success, failure):
    results_27.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


with checked("M18.27"):                                       # (1) the direction H_27, then grad_trace_fixed
    if not filled(H_27):
        print("⏳ Ex M18.27 : (1) pas encore fait (H_27).")
        raise StopCheck
    H_c27 = np.asarray(H_27, dtype=float)
    if H_c27.shape != X_27.shape:
        print(f"❌ Ex M18.27 : (1) H_27 doit avoir la forme de X_27, {X_27.shape} ; reçu {H_c27.shape}.")
        raise StopCheck
    report_27(not np.allclose(H_c27, H_c27.T), "(1) H_27 n'est pas symétrique",
              "(1) H_27 est symétrique : le long d'une direction symétrique, G et Gᵀ prévoient la même chose (fiche §8)")
    report_27(not check_along(trace_27, grad_trace(A_27, X_27), X_27, H_c27)[0],
              "(1) le long de H_27, check_along démasque grad_trace",
              "(1) le long de H_27, check_along laisse encore passer grad_trace : tire une direction au hasard")
    rng_27 = np.random.default_rng(5138)
    wrong_27 = []
    for shape_27 in ((2, 2), (3, 3), (2, 3), (1, 4)):        # A of shape (m, n), X of shape (n, m)
        A_c27, X_c27 = rng_27.normal(size=shape_27), rng_27.normal(size=shape_27[::-1])
        got_27 = grad_trace_fixed(A_c27.copy(), X_c27.copy())
        if got_27 is None or np.shape(got_27) != X_c27.shape or not np.allclose(np.asarray(got_27, dtype=float),
                                                                                 A_c27.T):
            wrong_27.append(f"A de forme {shape_27}, X de forme {shape_27[::-1]} : attendu un gradient de la forme de X, "
                            f"{X_c27.shape}, reçu " + ("None" if got_27 is None else f"la forme {np.shape(got_27)}"
                                                       if np.shape(got_27) != X_c27.shape else "d'autres nombres"))
    report_27(not wrong_27, "(1) grad_trace_fixed : quatre matrices A, carrées ou non", "(1) grad_trace_fixed, "
              + " ; ".join(wrong_27[:2]))
    parts_27.append(1)

with checked("M18.27"):                                       # (2) adjoints_27_fixed
    wrong_27 = []
    for point_27 in [tuple(POINT_27), *np.random.default_rng(5139).normal(size=(4, 2)).tolist()]:
        got_27 = adjoints_27_fixed(*point_27)
        want_27 = [f_27(point_27), *gradient_27(point_27)]
        if not (isinstance(got_27, tuple) and len(got_27) == 3 and np.allclose(np.asarray(got_27, dtype=float),
                                                                                want_27)):
            wrong_27.append(f"en {np.round(point_27, 3).tolist()} : attendu {np.round(want_27, 4).tolist()}, reçu "
                            f"{got_27!r}"[:160])
    report_27(not wrong_27, "(2) adjoints_27_fixed : la valeur et le gradient en cinq points",
              "(2) adjoints_27_fixed, " + " ; ".join(wrong_27[:2]))
    parts_27.append(2)

with checked("M18.27"):                                       # (3) the step of a reliable check
    if not filled(STEP_27):
        print("⏳ Ex M18.27 : (3) pas encore fait (STEP_27).")
        raise StopCheck
    h_27 = float(STEP_27)
    report_27(1e-6 <= h_27 <= 1e-4, f"(3) h = {h_27:g}, de l'ordre de 10⁻⁵ (fiche §8)",
              f"(3) h = {h_27:g} : choisis un pas de l'ordre de 10⁻⁵, entre 10⁻⁶ et 10⁻⁴, celui que conseillent la "
              f"fiche §8 et M05a pour une différence centrée en float64")
    rng_27 = np.random.default_rng(5139)
    cases_27 = [(POINT_27, DIRECTION_27)] + [(rng_27.normal(size=2), rng_27.normal(size=2)) for _ in range(4)]
    refused_27 = [p for p, d in cases_27 if not check_along(f_27, gradient_27(p), p, d, h=h_27)[0]]
    report_27(not refused_27, f"(3) avec h = {h_27:g}, la vérification accepte le gradient juste en cinq points",
              f"(3) avec h = {h_27:g}, la vérification refuse encore le gradient juste en {len(refused_27)} point(s) "
              f"sur 5 : un pas trop petit (les arrondis) ou trop grand (la troncature)")
    caught_27 = [p for p, d in cases_27 if not check_along(f_27, np.array(adjoints_27(*p)[1:]), p, d, h=h_27)[0]]
    report_27(len(caught_27) == len(cases_27), "(3) et elle démasque la passe du collègue, en cinq points",
              f"(3) avec ce pas, la vérification laisse passer la passe fausse du collègue en "
              f"{len(cases_27) - len(caught_27)} point(s)")
    parts_27.append(3)

if len(parts_27) == 3:
    verdict("M18.27", all(results_27), "la direction, les deux fonctions corrigées et le pas passent tous les cas.",
            "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pourquoi la direction $\mathbf{I}$ ne pouvait-elle pas voir l'erreur de `grad_trace` (que vaut $\sum_{ij}G_{ij}I_{ij}$, pour $\mathbf{G}$ et pour $\mathbf{G}^\top$) ? Pour quelles matrices $\mathbf{A}$ `grad_trace` aurait-elle raison ? Dans `adjoints_27`, quel chemin de $x$ vers $f$ l'adjoint incomplet oublie-t-il ? Avec $h = 10^{-13}$, de quel ordre est l'erreur d'arrondi de la différence centrée (fiche §8, M05a), et pourquoi dépasse-t-elle $10^{-6}$ ?

### Ex M18.28 — Tester une classe par ses propriétés : distributivité et dérivées connues des nombres duaux 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire des tests de propriété pour une classe de nombres duaux, et s'assurer qu'ils attrapent deux classes fausses.  
**Prérequis :** fiche §3 ; Ex M18.20 ; M17.22 · **Parcours :** C

Un **test de propriété** vérifie une loi que tout résultat juste respecte, sans connaître la réponse de chaque cas (M17.22). Écris deux fonctions de test dans le style de pytest (des `assert` avec un message qui dit le cas), qui reçoivent une **classe** de nombres duaux `Dual`, de même contrat que la tienne :
- `test_distributivity(Dual)` : avec un générateur de graine 5136 (`np.random.default_rng(5136)`, créé dans le test), tire au moins 20 triplets de duaux $u$, $v$, $w$ (valeurs et dérivées de loi normale), et vérifie $u(v + w) = uv + uw$, valeur **et** dérivée (`math.isclose(..., rel_tol=1e-9, abs_tol=1e-12)`), en calculant $u(v + w)$ d'abord ;
- `test_known_derivatives(Dual)` : en quelques points $t > 0$ (au moins trois), la dérivée calculée avec `Dual(t, 1.0)` de $\sin^2 t + \cos^2 t$ est nulle, celle de $e^{\ln t}$ vaut 1, et celle de $t^p$ vaut $p\,t^{p-1}$ pour quelques exposants $p$ (entiers, négatif, fractionnaire). Écris les carrés comme des **produits** (`s * s`, avec `s = t.sin()`) : le test doit passer par la multiplication.

La cellule fournie donne deux classes **fausses**, construites sur la référence : `BrokenDual`, dont le produit suit une règle fausse, et `AliasingDual`, dont le produit modifie son opérande de gauche au lieu de rendre un nouveau `Dual` (une erreur classique d'une classe Python). La vérification lance tes deux tests sur le `Dual` de la référence, où ils doivent **passer** ; sur `BrokenDual` et sur `AliasingDual`, où au moins un test doit **échouer** pour chacune, par un `assert` et pas par une autre erreur ; puis sur **ton** `Dual`.

In [ ]:
RefDual = ref_autodiff.Dual


class BrokenDual(RefDual):
    """A Dual whose product follows a wrong rule; every other operation is the reference's, and keeps the class."""

    def __mul__(self, other):
        o = other if isinstance(other, RefDual) else RefDual(other, 0.0)
        return BrokenDual(self.value * o.value, self.derivative * o.derivative)

    __rmul__ = __mul__


class AliasingDual(RefDual):
    """A Dual whose product changes its left operand in place and returns it, instead of a new Dual; every other
    operation is the reference's, and keeps the class."""

    def __mul__(self, other):
        o = other if isinstance(other, RefDual) else RefDual(other, 0.0)
        self.value, self.derivative = self.value * o.value, self.value * o.derivative + self.derivative * o.value
        return self

    __rmul__ = __mul__


def _keeping_class(name):
    """The method `name` of the reference, run on a plain copy of self, its result turned back into the class."""
    method = getattr(RefDual, name)

    def wrapper(self, *args):
        result = method(RefDual(self.value, self.derivative), *args)
        return type(self)(result.value, result.derivative) if isinstance(result, RefDual) else result
    return wrapper


for _cls in (BrokenDual, AliasingDual):
    for _name in ("__add__", "__radd__", "__sub__", "__rsub__", "__neg__", "__truediv__", "__rtruediv__", "__pow__",
                  "exp", "log", "sin", "cos"):
        setattr(_cls, _name, _keeping_class(_name))

In [ ]:
def test_distributivity(Dual):
    """u (v + w) = u v + u w, value and derivative, for at least 20 triples of duals drawn with the seed 5136."""
    raise NotImplementedError("test_distributivity n'est pas encore écrit (04_indices.md si tu bloques)")


def test_known_derivatives(Dual):
    """At a few points t > 0: (sin²t + cos²t)′ = 0 (squares written as products), (e^(ln t))′ = 1 and
    (t^p)′ = p t^(p − 1) for a few exponents p."""
    raise NotImplementedError("test_known_derivatives n'est pas encore écrit (04_indices.md si tu bloques)")

In [ ]:
def outcome_28(test, cls):
    """"" when test(cls) passes; "assert : …" when an assert fails; "erreur …" for another error (⏳ goes through)."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            test(cls)
        return ""
    except NotImplementedError:
        raise
    except AssertionError as exc:
        return "assert : " + (str(exc)[:140] or "un assert sans message")
    except Exception as exc:  # noqa: BLE001 - a crash is not a failed assert
        return f"erreur {type(exc).__name__} : {str(exc)[:120]}"


tests_28 = {"test_distributivity": test_distributivity, "test_known_derivatives": test_known_derivatives}
with checked("M18.28"):
    for name_28, test_28 in tests_28.items():
        if len(inspect.signature(test_28).parameters) != 1:
            raise TypeError(f"{name_28} prend un seul argument, la classe à tester : def {name_28}(Dual)")
    results_28 = []

    def report_28(ok, success, failure):
        results_28.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    for name_28, test_28 in tests_28.items():
        why_28 = outcome_28(test_28, ref_autodiff.Dual)
        report_28(not why_28, f"{name_28} passe sur le Dual de la référence",
                  f"{name_28} échoue sur le Dual de la référence, une classe juste : c'est le test qui est faux "
                  f"({why_28})")
    hints_28 = {"BrokenDual": "écris les carrés de sin² + cos² comme des produits (s * s), pas avec **",
                "AliasingDual": "dans la distributivité, calcule u * (v + w) d'abord, puis u * v + u * w avec les "
                                "MÊMES objets u, v, w"}
    for label_28, cls_28 in (("BrokenDual", BrokenDual), ("AliasingDual", AliasingDual)):
        outcomes_28 = {name: outcome_28(test, cls_28) for name, test in tests_28.items()}
        caught_28 = [name for name, why in outcomes_28.items() if why.startswith("assert")]
        crashed_28 = [f"{name} ({why})" for name, why in outcomes_28.items() if why.startswith("erreur")]
        report_28(caught_28 and not crashed_28, f"{label_28} est attrapée par {', '.join(caught_28)}",
                  f"{label_28} : " + (f"un test plante au lieu d'échouer sur un assert : {'; '.join(crashed_28)}"
                                      if crashed_28 else f"aucun de tes tests ne la voit ; {hints_28[label_28]}"))
    verdict("M18.28", all(results_28), "tes deux tests passent sur une classe juste et attrapent les deux classes "
                                       "fausses.", "relis les lignes ❌ ci-dessus.")

with checked("M18.28"):                                      # your own Dual
    for name_28, test_28 in tests_28.items():
        why_28 = outcome_28(test_28, mymath.autodiff.Dual)
        print(f"✅ {name_28} passe sur ton Dual" if not why_28 else
              f"❌ {name_28} échoue sur ton Dual ({why_28}) : si tes tests passent sur la référence, l'erreur est dans "
              f"ta classe (lance aussi les tests de M18.20)")

**Après la vérification**, dans ta copie : lequel de tes tests attrape chaque classe fausse, et pourquoi l'autre ne la voit pas ? Montre que toute règle du produit de la forme $(a + b\varepsilon)(c + d\varepsilon) = ac + (\alpha\,ad + \beta\,bc + \gamma\,bd)\,\varepsilon$, quels que soient $\alpha$, $\beta$, $\gamma$, respecte la distributivité : qu'est-ce que ce test vérifie vraiment ? Une classe fausse pourrait-elle passer tes deux tests ? Cherche une erreur qui touche `sin` et `cos` à la fois et garde $\sin^2 + \cos^2$ constant. Que verraient alors les tests de `mymath`, qui comparent à SymPy ?

### Ex M18.29 — Défi : dériver une boucle, la méthode de Newton pour √a, en mode direct 🏆 ★★★ ⏱️ 30 min
**Objectif :** dériver un algorithme itératif en faisant passer des nombres duaux à travers sa boucle, et atteindre une précision fixée en un nombre de pas limité.  
**Prérequis :** fiche §3 ; Ex M18.20 · **Parcours :** C

Une boucle se dérive aussi en mode direct : les nombres duaux traversent chaque itération. La **méthode de Newton** pour l'équation $x^2 = a$ ($a > 0$) répète $x \leftarrow \frac12\left(x + \frac{a}{x}\right)$, et $x$ converge vers $\sqrt{a}$ très vite : dès qu'on en est assez près, le nombre de chiffres justes double à chaque pas. Lancée avec `a = Dual(a, 1.0)`, la partie en $\varepsilon$ de l'itéré est sa dérivée par rapport à $a$, qui tend vers $\frac{1}{2\sqrt{a}}$ : la dérivée de la racine, sans jamais l'avoir écrite.

**L'objectif chiffré.** Écris `sqrt_and_derivative(a)`, qui rend le couple de flottants $(\sqrt{a}, \frac{1}{2\sqrt{a}})$ pour un flottant $a > 0$ :
- par la méthode de Newton, chaque pas étant un appel de `newton_step_29(x, a)` (cellule fournie : elle fait un pas, et compte ses appels), avec $a$ un `Dual` de **ta** classe (`mymath.autodiff.Dual(a, 1.0)`) ;
- **en 8 pas au plus** par appel ;
- sans racine toute faite : ni `math.sqrt`, ni `np.sqrt`, ni `** 0.5` (aucune puissance d'exposant décimal ou fractionnaire), ni `pow`, `exp` ou `log` (la vérification lit le code de ta fonction) ;
- avec une erreur relative d'au plus $10^{-12}$ sur chacun des deux nombres, pour chacune des 200 valeurs de `A_VALUES_29`, régulièrement espacées en échelle logarithmique de $10^{-2}$ à $10^4$.

Un départ $x_0 = 1$ ne suffit pas : pour $a = 10^4$, les premiers pas ne font guère que diviser $x$ par 2, et 8 pas sont loin du compte. Trouve un meilleur départ, avec des opérations permises.

La vérification appelle ta fonction sur les 200 valeurs, compte les pas, mesure les erreurs, lit ton code, puis enregistre si l'objectif est atteint.

In [ ]:
A_VALUES_29 = np.geomspace(1e-2, 1e4, 200)      # 200 values of a, evenly spaced on a log scale
NEWTON_29 = {"calls": 0, "plain": 0}            # the calls of newton_step_29, and those whose a is not a dual


def newton_step_29(x, a):
    """One step of Newton's method for x² = a: (x + a / x) / 2, for numbers or duals. Counts its calls."""
    NEWTON_29["calls"] += 1
    if not hasattr(a, "derivative"):
        NEWTON_29["plain"] += 1
    return (x + a / x) / 2


FORBIDDEN_29 = {"sqrt", "isqrt", "cbrt", "pow", "power", "float_power", "exp", "exp2", "expm1", "log", "log2", "log10",
                "log1p", "hypot"}


def forbidden_29(function):
    """The forbidden ways to a square root found in the source of function: the names of FORBIDDEN_29, and the powers
    whose exponent holds a float or a division (** 0.5, ** (1 / 2)). None when the source cannot be read."""
    try:
        tree = ast.parse(textwrap.dedent(inspect.getsource(function)))
    except Exception:  # noqa: BLE001 - OSError, SyntaxError, a tokenize error: the source cannot be read here
        return None
    found = set()
    for node in ast.walk(tree):
        name = node.id if isinstance(node, ast.Name) else node.attr if isinstance(node, ast.Attribute) else None
        if name in FORBIDDEN_29:
            found.add(name)
        if isinstance(node, (ast.BinOp, ast.AugAssign)) and isinstance(node.op, ast.Pow):
            exponent = node.right if isinstance(node, ast.BinOp) else node.value
            if any((isinstance(part, ast.Constant) and isinstance(part.value, float))
                   or (isinstance(part, ast.BinOp) and isinstance(part.op, ast.Div)) for part in ast.walk(exponent)):
                found.add("** " + ast.unparse(exponent))
    return sorted(found)

In [ ]:
def sqrt_and_derivative(a):
    """(√a, 1/(2√a)) for a float a > 0, by Newton's method run on dual numbers (a as Dual(a, 1.0)): at most 8 calls
    of newton_step_29, and no square root (math.sqrt, np.sqrt, ** 0.5, pow, exp, log)."""
    raise NotImplementedError("sqrt_and_derivative n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M18.29"):
    found_29 = forbidden_29(sqrt_and_derivative)
    worst_value_29 = worst_derivative_29 = 0.0
    most_steps_29 = plain_29 = no_step_29 = 0
    for a_29 in A_VALUES_29:
        NEWTON_29.update(calls=0, plain=0)
        result_29 = sqrt_and_derivative(float(a_29))
        if not (isinstance(result_29, tuple) and len(result_29) == 2):
            print(f"❌ Ex M18.29 : sqrt_and_derivative({a_29:g}) doit rendre un couple (valeur, dérivée), reçu "
                  f"{result_29!r}"[:200])
            raise StopCheck
        value_29, derivative_29 = float(result_29[0]), float(result_29[1])
        root_29 = math.sqrt(a_29)                            # the check may use a square root
        errors_29 = (abs(value_29 / root_29 - 1), abs(derivative_29 * 2 * root_29 - 1))
        errors_29 = [e if math.isfinite(e) else math.inf for e in errors_29]
        worst_value_29, worst_derivative_29 = max(worst_value_29, errors_29[0]), max(worst_derivative_29, errors_29[1])
        most_steps_29 = max(most_steps_29, NEWTON_29["calls"])
        plain_29 += NEWTON_29["plain"]
        no_step_29 += NEWTON_29["calls"] == 0
    results_29 = []

    def report_29(ok, success, failure):
        results_29.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    if found_29 is None:
        print("⚠️ le code de ta fonction n'a pas pu être lu ici : les opérations interdites ne sont pas vérifiées.")
    else:
        report_29(not found_29, "aucune racine toute faite dans ton code",
                  f"ton code utilise une opération interdite : {', '.join(found_29)}")
    report_29(no_step_29 == 0 and plain_29 == 0, "chaque appel passe par newton_step_29, avec a en nombre dual",
              f"chaque appel doit faire ses pas par newton_step_29, avec a = Dual(a, 1.0) de ta classe : {no_step_29} "
              f"appel(s) sans aucun pas, {plain_29} pas avec un a qui n'est pas un Dual")
    report_29(most_steps_29 <= 8, f"au plus {most_steps_29} pas de Newton par appel (8 permis)",
              f"jusqu'à {most_steps_29} pas de Newton pour une valeur de a : 8 au plus ; il faut un meilleur départ")
    report_29(worst_value_29 <= 1e-12, f"√a à {worst_value_29:.1e} près en relatif (objectif : 1e-12)",
              f"√a n'est juste qu'à {worst_value_29:.1e} près en relatif, pour l'une des 200 valeurs (objectif : 1e-12)")
    report_29(worst_derivative_29 <= 1e-12, f"1/(2√a) à {worst_derivative_29:.1e} près en relatif (objectif : 1e-12)",
              f"la dérivée n'est juste qu'à {worst_derivative_29:.1e} près en relatif (objectif : 1e-12)")
    for a_29 in (0.01, 2.0, 1e4):
        value_29, derivative_29 = sqrt_and_derivative(a_29)
        print(f"   a = {a_29:g} : √a ≈ {value_29:.15g} (math.sqrt : {math.sqrt(a_29):.15g}) ; dérivée ≈ "
              f"{derivative_29:.15g} (1/(2√a) : {0.5 / math.sqrt(a_29):.15g})")
    wbm.check("M18.29", all(results_29), computed=True)

**Dans ta copie** : dérive l'itération $x_{k+1} = \frac12\left(x_k + \frac{a}{x_k}\right)$ par rapport à $a$ et montre que $x_{k+1}' = \frac12\left(1 - \frac{a}{x_k^2}\right)x_k' + \frac{1}{2x_k}$. Quand $x_k$ est proche de $\sqrt{a}$, que vaut le facteur devant $x_k'$, et pourquoi la dérivée converge-t-elle alors vers $\frac{1}{2\sqrt{a}}$ quelle que soit l'erreur sur $x_k'$ ? Pourquoi un départ constant, de dérivée nulle, convient-il ? Combien de pas te faut-il au plus, et pourquoi ?

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu écrire le graphe de calcul d'une formule, et dériver sa sortie par la somme sur les chemins ?
2. Sais-tu propager des dérivées en mode direct, à la main et avec des nombres duaux ?
3. Sais-tu faire une passe arrière à la main, en accumulant les adjoints d'un nœud qui sert plusieurs fois ?
4. Sais-tu choisir le mode selon le nombre d'entrées et de sorties, et estimer son coût ?
5. Sais-tu calculer le gradient d'une fonction d'une matrice, de la forme de la matrice, et le vérifier le long d'une direction tirée au hasard ?

**La suite** : le chapitre 18 de workbookIA, qui applique le mode inverse au graphe d'un réseau de neurones (la rétropropagation), puis M19 (hessienne, conditionnement et convergence), juste avant le chapitre 19 de workbookIA.